# ✋ 動手做：用辦公室資料畫五張圖（由淺至深）

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/roberthsu2003/PythonForDataAnalysis/blob/master/%E7%9C%8B%E6%87%82%E8%B3%87%E6%96%99%E8%88%87%E9%81%B8%E5%B0%8D%E5%9C%96%E8%A1%A8/README.ipynb)

這本筆記本把「看懂資料與選對圖表」的 8 堂課**完整走一遍**。你不需要自己寫程式：每張圖都附有「給 AI 的指令」和可直接執行的參考程式，請把力氣花在**看懂每一步為什麼這樣做**。

## 🪜 五張圖的順序：每一張只多學一件事

| 順序 | 問題 | 類別 | 圖表 | 多學的一件事 |
| :---: | :--- | :--- | :--- | :--- |
| 1 | 哪個部門每人平均支出最高？ | 比較 | 垂直長條圖 | 數值軸從 0 開始、標上單位 |
| 2 | 每月支出怎麼變化？ | 趨勢 | 折線圖 | 時間由早到晚排序才能連線 |
| 3 | 各採購類別各占多少？ | 組成 | 百分比長條圖 | 先講清楚**分母** |
| 4 | 同仁通勤時間大多多久？ | 分布 | 直方圖 | 平均數和中位數要一起看 |
| 5 | 預估工時和實際工時有關嗎？ | 關係 | 散佈圖 | **相關不等於因果** |

## 🤖 怎麼使用
1. 先執行下方「第 0 步」（點左邊的 ▶）。
2. 每張圖先讀上方的**思考表格**，自己想想答案。
3. 可以把「給 AI 的指令」貼給 Colab 的 Gemini，再和參考程式比較；或直接執行參考程式。
4. 看完圖，逐項完成「✅ 檢查一下」。

In [ ]:
# 第 0 步：下載練習資料並設定繁體中文字型（請先點擊左側 ▶ 執行此儲存格）
import os
from pathlib import Path
from urllib.request import urlretrieve

file_path = Path('五類圖表_練習資料.xlsx')
if not file_path.exists():
    local = Path('學生素材') / file_path.name          # 在本機 repo 開啟時直接使用旁邊的檔案
    if local.exists():
        file_path = local
    else:
        urlretrieve('https://raw.githubusercontent.com/roberthsu2003/PythonForDataAnalysis/master/%E7%9C%8B%E6%87%82%E8%B3%87%E6%96%99%E8%88%87%E9%81%B8%E5%B0%8D%E5%9C%96%E8%A1%A8/%E5%AD%B8%E7%94%9F%E7%B4%A0%E6%9D%90/%E4%BA%94%E9%A1%9E%E5%9C%96%E8%A1%A8_%E7%B7%B4%E7%BF%92%E8%B3%87%E6%96%99.xlsx', file_path)
print('📂 資料檔：', file_path)

%pip install -q mplfonts
import glob
import mplfonts
from matplotlib import font_manager
from mplfonts import use_font

for f in glob.glob(os.path.join(os.path.dirname(mplfonts.__file__), "fonts", "*.[ot]tf")):
    font_manager.fontManager.addfont(f)
use_font("Noto Sans CJK SC")
print("✅ 環境初始化完成！")

---

## 先看懂資料（第 7 課・第一階）

> 先不要畫圖。這個 Excel 有 5 張工作表，**每張表一列代表的東西都不一樣**，先一張一張看清楚。

📋 **給 AI 的指令**：
```text
請讀取「五類圖表_練習資料.xlsx」的所有工作表。每張工作表顯示列數、欄位名稱與前 3 列，
並說明「一列代表什麼」，以及每一欄是指標、條件還是標識。先不要畫圖。
```

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

sheets = pd.read_excel(file_path, sheet_name=None)
one_row = {'比較_部門支出': '一個部門', '組成_採購類別': '一個採購類別', '分布_通勤時間': '一位員工',
           '關係_專案工時': '一個專案', '趨勢_每月支出': '一個月份'}
for name, table in sheets.items():
    print(f"\n=== {name}：{len(table)} 列，一列代表「{one_row.get(name, '？')}」 ===")
    display(table.head(3))

✅ **檢查一下**：
- [ ] 能說出每張表一列代表什麼嗎？
- [ ] 「員工代號」「專案編號」是**標識**——只能數有幾筆，不能加總或平均。

---

## 圖 1｜比較：哪個部門每人平均支出最高？

> 🎯 **問題**：哪個部門的每人平均支出最高？和最低的差多少？

| 思考步驟 | 這張圖的答案 |
| :--- | :--- |
| 第 1 課・欄位貼標籤 | `部門`＝條件（類別）；`每人平均支出（元）`＝指標 |
| 第 4 課・問題類別 | 比較（「最高」「差多少」） |
| 第 5～6 課・選圖 | 5 個部門、名稱短 ➔ **垂直長條圖** |
| 🆕 這張圖多學的一件事 | 長條圖的數值軸**一定從 0 開始**，否則會誇大差距（地雷 1） |

📋 **給 AI 的指令（第 7 課・第二階）**：
```text
請用「比較_部門支出」工作表回答：哪個部門的每人平均支出最高？
1. 先依每人平均支出由高到低排序，顯示數字表。
2. 畫垂直長條圖：X 軸放部門、Y 軸放每人平均支出（元），Y 軸從 0 開始，柱子上標註金額。
3. 最高的部門用醒目顏色。另外並排畫一張「Y 軸不從 0 開始」的錯誤示範，讓我比較差別。
```

In [ ]:
comparison = sheets['比較_部門支出'].sort_values('每人平均支出（元）', ascending=False)
display(comparison)

top = comparison.iloc[0]['部門']
colors = ['#e74c3c' if d == top else '#3498db' for d in comparison['部門']]
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, bottom, title in [(axes[0], 0, '正確：數值軸從 0 開始'), (axes[1], 100000, '地雷：數值軸從 100,000 開始')]:
    ax.bar(comparison['部門'], comparison['每人平均支出（元）'], color=colors, width=0.55)
    ax.set_ylim(bottom, comparison['每人平均支出（元）'].max() * 1.12)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel('部門')
    ax.set_ylabel('每人平均支出（元）')
    ax.ticklabel_format(axis='y', style='plain')
for x, y in zip(comparison['部門'], comparison['每人平均支出（元）']):
    axes[0].text(x, y * 1.01, f'{y:,}', ha='center')
plt.suptitle(f'各部門每人平均支出（最高：{top}）', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

high, low = comparison['每人平均支出（元）'].max(), comparison['每人平均支出（元）'].min()
print(f"💡 最高 {high:,} 元、最低 {low:,} 元，相差 {(high - low) / low:.0%}。右圖看起來差了好幾倍，就是截斷數值軸造成的錯覺。")

✅ **檢查一下**：
- [ ] 表格和左圖都顯示同一個部門最高嗎？
- [ ] 左圖的數值軸從 0 開始、有單位「元」嗎？
- [ ] 能說出右圖為什麼會讓人誤會嗎？

🚫 **不能推論**：每人平均支出高，不代表該部門「浪費」——資料沒有說明支出的用途。

---

## 圖 2｜趨勢：全公司每月支出怎麼變化？

> 🎯 **問題**：2025 年全公司每月實際支出是增加還是減少？哪個月最高？

| 思考步驟 | 這張圖的答案 |
| :--- | :--- |
| 第 1 課・欄位貼標籤 | `月份`＝條件（時間）；`全公司實際支出（元）`＝指標 |
| 第 4 課・問題類別 | 趨勢（「怎麼變化」「哪個月」） |
| 第 5～6 課・選圖 | 12 個時間點 ➔ **折線圖** |
| 🆕 這張圖多學的一件事 | 只有**時間**可以連線，而且要先**由早到晚排序**（地雷 3） |

📋 **給 AI 的指令（第 7 課・第二階）**：
```text
請用「趨勢_每月支出」工作表回答：全公司每月支出怎麼變化？
1. 先把月份由早到晚排序。
2. 畫折線圖：X 軸放月份（標成 1月～12月）、Y 軸放實際支出（元），每個點加上圓點。
3. 標出最高與最低的月份與金額。
```

In [ ]:
trend = sheets['趨勢_每月支出'].sort_values('月份')
labels = [f'{m.month}月' for m in trend['月份']]
values = trend['全公司實際支出（元）']
hi, lo = values.idxmax(), values.idxmin()

plt.figure(figsize=(10, 4.8))
plt.plot(labels, values, marker='o', color='#3498db', linewidth=2)
for idx, color in [(hi, '#e74c3c'), (lo, '#2c3e50')]:
    month = f"{trend.loc[idx, '月份'].month}月"
    plt.annotate(f'{month}\n{values[idx]:,}', (month, values[idx]), textcoords='offset points',
                 xytext=(0, 10), ha='center', color=color, fontweight='bold')
plt.ylim(0, values.max() * 1.2)
plt.ticklabel_format(axis='y', style='plain')
plt.title('2025 年全公司每月實際支出', fontsize=14, fontweight='bold')
plt.xlabel('月份')
plt.ylabel('實際支出（元）')
plt.grid(linestyle='--', alpha=0.4)
plt.show()
print(f"💡 最高 {trend.loc[hi, '月份'].month} 月（{values[hi]:,} 元），最低 {trend.loc[lo, '月份'].month} 月（{values[lo]:,} 元）。")

✅ **檢查一下**：
- [ ] X 軸是不是從 1 月排到 12 月？
- [ ] 最高、最低月份的數字和表格一樣嗎？
- [ ] 想一想：如果把「部門」也連成折線，為什麼不對？

🚫 **不能推論**：只有一年資料，不能說「每年都會這樣」，也不能預測明年。

---

## 圖 3｜組成：各採購類別各占多少？

> 🎯 **問題**：全年採購金額中，各採購類別各占幾成？

| 思考步驟 | 這張圖的答案 |
| :--- | :--- |
| 第 1 課・欄位貼標籤 | `採購類別`＝條件（類別）；`採購金額（元）`＝指標 |
| 第 4 課・問題類別 | 組成（「各占幾成」） |
| 第 5～6 課・選圖 | 5 個類別（超過 4 個）➔ **百分比長條圖**，不用圓餅圖（地雷 2） |
| 🆕 這張圖多學的一件事 | 百分比一定要先講清楚**分母**是什麼（地雷 4） |

📋 **給 AI 的指令（第 7 課・第二階）**：
```text
請用「組成_採購類別」工作表回答：各採購類別各占全年採購金額多少？
1. 先算出全年採購總額（分母），再算每個類別的百分比，由大到小排序。
2. 畫一根水平的百分比長條圖（總和 100%），每段標示類別名稱與百分比。
3. 標題寫明「分母＝全年採購總額 X 元」。
```

In [ ]:
composition = sheets['組成_採購類別'].sort_values('採購金額（元）', ascending=False).copy()
total = composition['採購金額（元）'].sum()                      # 分母：全年採購總額
composition['占比（%）'] = (composition['採購金額（元）'] / total * 100).round(1)
display(composition)

fig, ax = plt.subplots(figsize=(11, 2.4))
left = 0
for (_, row), color in zip(composition.iterrows(), plt.cm.Set2.colors):
    ax.barh(0, row['占比（%）'], left=left, color=color)
    ax.text(left + row['占比（%）'] / 2, 0, f"{row['採購類別']}\n{row['占比（%）']}%", ha='center', va='center')
    left += row['占比（%）']
ax.set_xlim(0, 100)
ax.set_yticks([])
ax.set_xlabel('占全年採購總額（%）')
ax.set_title(f'採購金額組成（分母＝全年採購總額 {total:,} 元）', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
print('💡 5 個類別的占比都在 1～3 成之間，切成圓餅圖很難分辨大小；百分比長條圖比較容易比較。')

✅ **檢查一下**：
- [ ] 所有百分比加起來是 100% 嗎？
- [ ] 標題有寫出分母（全年採購總額）嗎？
- [ ] 能說出為什麼這裡不用圓餅圖嗎？

🚫 **不能推論**：占比最高的類別，不代表它「太貴」或「該砍預算」——資料沒有說明每一筆採購的必要性。

---

## 圖 4｜分布：同仁通勤時間大多多久？

> 🎯 **問題**：60 位同仁的通勤時間大多落在幾分鐘？有沒有特別遠的？

| 思考步驟 | 這張圖的答案 |
| :--- | :--- |
| 第 1 課・欄位貼標籤 | `通勤時間（分鐘）`＝指標；`員工代號`＝標識（不參與計算） |
| 第 4 課・問題類別 | 分布（「大多」「有沒有特別」） |
| 第 5～6 課・選圖 | 一個指標、60 筆 ➔ **直方圖** |
| 🆕 這張圖多學的一件事 | 平均數會被極端值拉動，要和**中位數**一起看 |

📋 **給 AI 的指令（第 7 課・第二階）**：
```text
請用「分布_通勤時間」工作表回答：同仁通勤時間大多落在幾分鐘？
1. 先算出人數、平均數、中位數、最小值、最大值。
2. 畫直方圖：X 軸為通勤時間（分鐘，每 10 分鐘一組）、Y 軸為人數，並畫出平均數線與中位數線。
3. 標題寫出人數 n，並用數字描述大多數人集中的區間。
```

In [ ]:
commute = sheets['分布_通勤時間']['通勤時間（分鐘）']
display(commute.agg(['count', 'mean', 'median', 'min', 'max']).round(1).to_frame('通勤時間（分鐘）'))

plt.figure(figsize=(9, 4.8))
plt.hist(commute, bins=range(10, 101, 10), color='#3498db', edgecolor='white')
plt.axvline(commute.mean(), color='#e74c3c', linestyle='--', linewidth=2, label=f'平均數 {commute.mean():.1f} 分鐘')
plt.axvline(commute.median(), color='#2c3e50', linestyle=':', linewidth=2, label=f'中位數 {commute.median():.1f} 分鐘')
plt.title(f'同仁通勤時間分布（n = {len(commute)}）', fontsize=14, fontweight='bold')
plt.xlabel('通勤時間（分鐘，每 10 分鐘一組）')
plt.ylabel('人數')
plt.legend()
plt.show()
q1, q3 = commute.quantile([0.25, 0.75])
print(f"💡 一半的同仁通勤時間在 {q1:.0f}～{q3:.0f} 分鐘之間；最遠的 {commute.max()} 分鐘把平均數往上拉了一點。")

✅ **檢查一下**：
- [ ] 直方圖的柱子是不是緊緊相連（代表連續的區間）？
- [ ] 平均數和中位數差多少？哪一個比較能代表「一般同仁」？
- [ ] 員工代號有沒有被拿去計算？

🚫 **不能推論**：只有通勤時間，不能說「通勤越久的人越不滿意」——資料裡沒有滿意度。

---

## 圖 5｜關係：預估工時和實際工時有關嗎？

> 🎯 **問題**：預估工時越長的專案，實際工時是不是也越長？

| 思考步驟 | 這張圖的答案 |
| :--- | :--- |
| 第 1 課・欄位貼標籤 | `預估工時`、`實際工時`＝兩個指標；`專案編號`＝標識 |
| 第 4 課・問題類別 | 關係（「越…越…」） |
| 第 5～6 課・選圖 | 兩個數值 ➔ **散佈圖**，點不連線 |
| 🆕 這張圖多學的一件事 | 看得出關聯，**不代表一個造成另一個**（地雷 5） |

📋 **給 AI 的指令（第 7 課・第二階）**：
```text
請用「關係_專案工時」工作表回答：預估工時和實際工時有沒有關聯？
1. 畫散佈圖：X 軸為預估工時、Y 軸為實際工時（小時），每個點代表一個專案，點不要連線。
2. 加上一條「預估＝實際」的虛線參考線，並計算相關係數 r 顯示在標題。
3. 統計有幾個專案實際工時超過預估，並說明這只是關聯、不是因果。
```

In [ ]:
projects = sheets['關係_專案工時']
r = projects['預估工時'].corr(projects['實際工時'])
over = (projects['實際工時'] > projects['預估工時']).sum()

plt.figure(figsize=(6.5, 6))
plt.scatter(projects['預估工時'], projects['實際工時'], color='#2ecc71', edgecolors='#27ae60', s=55)
limit = [0, max(projects['預估工時'].max(), projects['實際工時'].max()) * 1.1]
plt.plot(limit, limit, color='gray', linestyle='--', label='預估 = 實際')
plt.xlim(limit)
plt.ylim(limit)
plt.title(f'預估工時與實際工時（r = {r:.2f}，n = {len(projects)}）', fontsize=13, fontweight='bold')
plt.xlabel('預估工時（小時）')
plt.ylabel('實際工時（小時）')
plt.legend()
plt.grid(linestyle='--', alpha=0.4)
plt.show()
print(f"💡 r = {r:.2f}，兩者高度正相關；{over} / {len(projects)} 個專案實際工時超過預估（點在虛線上方）。")

✅ **檢查一下**：
- [ ] 點有沒有被連成線？
- [ ] 虛線上方的點代表什麼意思？
- [ ] 能用一句話說明「相關不等於因果」嗎？

🚫 **不能推論**：「預估得長，所以做得久」是因果說法；大專案本來就估得長、也做得久，兩者都受「專案規模」影響。

---

## 🎉 完成了！回顧五張圖

| 圖 | 類別 | 圖表 | 你學到的一件事 |
| :---: | :--- | :--- | :--- |
| 1 | 比較 | 垂直長條圖 | 數值軸從 0 開始 |
| 2 | 趨勢 | 折線圖 | 只有時間能連線，要先排序 |
| 3 | 組成 | 百分比長條圖 | 先講清楚分母；類別多不用圓餅圖 |
| 4 | 分布 | 直方圖 | 平均數和中位數一起看 |
| 5 | 關係 | 散佈圖 | 相關不等於因果 |

## 🗣️ 第 8 課｜選一張圖，寫出可信的結論

從上面的五張圖任選一張，先自己寫出四句話：① 觀察到什麼；② 哪些數字是證據，能否回表格核對；③ 資料涵蓋範圍與不能推論的事；④ 下一步需要查什麼。請勿只請 AI 代寫。寫完後可以用[第 8 課的檢查指令](./看完圖怎麼說出可信的結論/README.md#再用-ai-幫你挑錯)，請 AI 找出你可能說得太滿的地方。

例如選圖 1：資訊部每人平均支出為 145,000 元、行政部為 120,000 元，差額 25,000 元。請回工作表確認這些數字，再想想支出高低是否足以判斷效率。

## 🚀 接下來
- **挑戰**：打開 [範例_陌生資料挑戰_空氣品質.ipynb](./學生素材/範例_陌生資料挑戰_空氣品質.ipynb)，用一份沒看過的真實資料，自己走完整套流程。
- **下一章**：[AI 協作資料分析｜14 個循序實作範例](../AI協作資料分析_12個範例/README.md)——在 Colab 裡實際指揮 AI 完成從讀檔、清理到報表的完整分析。